<a href="https://colab.research.google.com/github/danielchin-ck/ADALL_github/blob/main/Project/6096089W_CDA1C03_ADALL_Project_2026-Ver2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Libraries used in this notebook
import os
import shutil
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Modelling libraries
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

# Make wide tables easier to read in Colab
pd.set_option('display.max_columns', 100)

In [ ]:
import os
from google.colab import userdata

token = userdata.get('GITHUB_TOKEN')
assert token, 'Missing Colab Secret: GITHUB_TOKEN'

os.environ['GITHUB_TOKEN'] = token
os.environ['GITHUB_USER'] = 'danielchin-ck'
os.environ['GITHUB_REPO'] = 'ADALL_github'
os.environ['GITHUB_EMAIL'] = 'danielchin.ck@gmail.com'

print('GitHub settings loaded. Token is not printed.')

In [6]:
RUN_API_CELLS = True
OPENAI_MODEL = 'gpt-5.4-nano'
client = None

if RUN_API_CELLS == True:
    from google.colab import userdata
    from openai import OpenAI

    api_key = userdata.get('OPENAI_API_KEY')
    client = OpenAI(api_key=api_key)
    print('OpenAI client is ready.')
else:
    print('Manual chatbot mode. Copy the prompts when they appear.')



OpenAI client is ready.


In [5]:
#Version 1
bp_prompt = f"""
You are a Data Scientist of specialized experience in tree-based regression models (XGBoost, LightGBM, Random Forest) for actuarial and real-estate forecasting.
Your expertise is translating ambiguous socio-economic problems into rigorous ML objectives with clear business impact.

CONTEXT:
Mr. Chin faces a criticial retirement challenge: his Executive Apartment HDB flat is depreciating through lease decay, but a new MRT station near his flat will be completed after 2029,
potentially boosting its value. He needs to identify the optimal year within the next 5 to 10 years to downsize - balancing these opposite forces.

YOUR TASK:
Craft a complete ML modelling brief that addresses the following:

1. BUSINESS PROBLEM DEFINITION
   •	Describe the business problem you aim to address. Explain why it matters and who is affected.

2. PRIMARY BENEFICIARY PERSONA
   •	Identify the key beneficiary of the ML solution. Create a simple persona that captures details important for this project,
   such as the person’s goals, pain points, behaviours and how they would use the solution.

3. JTBD-FRAMED MODELLING OBJECTIVES
   •	Use the Job-To-Be-Done (JTBD) framework to shape your modelling objectives. State the job the user is trying to complete,
   and define your target variable and likely predictors based on this job.

FORMAT:
Respond in point forms

CONSTRAINTS:
Keep it concise and to the point.
Limit each section to maximum 200 characters per line.
"""

if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=bp_prompt
    )
    print('\nLLM response:')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')



LLM response:
- **1) BUSINESS PROBLEM DEFINITION**
  - **Problem:** Forecast best downsizing year (next 5–10y) for HDB Exec apartment.
  - **Why it matters:** Flat value decays via lease term + may rise after new MRT completion.
  - **Impact:** Helps avoid overpaying on timing; reduces retirement risk for household cashflow.
  - **Affected:** Mr. Chin + co-owners; retirement planning; potential buyers/estate liquidity.

- **2) PRIMARY BENEFICIARY PERSONA**
  - **Persona:** “Kian Chin, 55–62, nearing retirement, owns Executive HDB.”
  - **Goal:** Choose downsizing year to maximize proceeds (or minimize loss) while maintaining housing stability.
  - **Pain points:** Unclear MRT price lift timing; uncertainty on lease decay vs transit premium.
  - **Behaviour:** Uses online price trends; waits for “right time”; needs a data-backed decision rule.
  - **Use:** Inputs address/lease/financial constraints → gets recommended year range + rationale.

- **3) JTBD-FRAMED MODELLING OBJECTIVES**
  

In [8]:
#Version 2
bp_prompt = f"""
You are a Data Scientist of specialized experience in tree-based regression models (XGBoost, LightGBM, Random Forest) for actuarial and real-estate forecasting.
Your expertise is translating ambiguous socio-economic problems into rigorous ML objectives with clear business impact.

CONTEXT:
Mr. Chin faces a criticial retirement challenge: his 36 years old Executive Apartment HDB flat is depreciating through lease decay, but a new MRT station near his flat will be completed after 2029,
potentially boosting its value. He needs to identify the optimal year within the next 5 to 10 years to downsize - balancing these opposite forces.

YOUR TASK:
Craft a complete ML modelling brief that addresses the following:

1. BUSINESS PROBLEM DEFINITION
   •	Describe the business problem you aim to address. Explain why it matters and who is affected.

2. PRIMARY BENEFICIARY PERSONA
   •	Identify the key beneficiary of the ML solution. Create a simple persona that captures details important for this project,
   such as the person’s goals, pain points, behaviours and how they would use the solution.

3. JTBD-FRAMED MODELLING OBJECTIVES
   •	Use the Job-To-Be-Done (JTBD) framework to shape your modelling objectives. State the job the user is trying to complete,
   and define your target variable and likely predictors based on this job.

FORMAT:
Respond in point forms

CONSTRAINTS:
Keep it concise and to the point.
Limit each section to maximum 200 characters per line.
"""

if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=bp_prompt
    )
    print('\nLLM response:')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')



LLM response:
- **1) Business Problem Definition**
  - Optimize best downsize timing (year) for an HDB Executive Apartment lease decay vs MRT value uplift.
  - Affects Mr. Chin’s retirement capital planning and housing affordability risk.
  - Deliver decision support: “Downsize year” choice within next 5–10 yrs.

- **2) Primary Beneficiary Persona**
  - **Persona:** 36yo owner-occupier of Exec Apartment HDB, planning retirement in 10–20 yrs.
  - **Goal:** Sell, downsize, and fund retirement while minimizing loss/overpay risk.
  - **Pain points:** Lease decay drag; uncertainty about MRT-driven uplift timing; limited time to analyze.
  - **Usage:** Inputs property + household + timeline; receives best year + expected value range.

- **3) JTBD-Framed Modelling Objectives**
  - **JTBD:** “Help me choose the year to sell/downsize so my net proceeds + affordability are maximized.”
  - **Target variable (regression):**
    - Predict **HDB resale price** (or **net proceeds**) by candidate yea

In [6]:
prepare_prompt = f"""
You are a Data Scientist of specialized experience in tree-based regression models (XGBoost, LightGBM, Random Forest) for actuarial and real-estate forecasting.

CONTEXT:
When Mr. Chin’s HDB Executive Apartment in Pasir Ris approaches the post-MRT era (2030 onwards) and he need to fund his retirement, he want to identify the best year within 2030–2036 to downsize, so that he can maximize his net sale proceeds by capturing the Cross Island Line MRT premium before lease decay erodes his flat's value.“

Target Variable (Regression)
Primary Target:
Projected Net Sale Proceeds (selling price) for each candidate year t, where t ∈ [2030, 2031, ..., 2036].

Key Assumptions:
✅ Nominal proceeds only (no discounting)
✅ No P10/P50 guardrails or probabilistic optimization
✅ MRT operational by 2030 (no time covariates needed)
Per Straits Times (23 Mar 2026), MRT premium decay of $54,027 per 100m applies only beyond 400m. Since Mr. Chin's flat is at 200m, I have applied the full premium $54,027 with no decay.

Predictors (Feature Set):
Property Attributes: Remaining lease (months/years), flat age, floor level, unit size (m²), town (Pasir Ris), precinct proxies, flat type
Location/MRT: Distance to MRT (200m), baseline accessibility, MRT premium
Market Dynamics: Historical resale price index (town-level), recent transacted m² in Pasir Ris, resale supply trends

Excluded Predictors:
❌ Time covariates (year index t, event indicators for MRT phases) — MRT operational by 2030
❌ Macro variables (interest rates, wage growth, inflation) — outside scope

Dataset: 	https://raw.githubusercontent.com/danielchin-ck/ADALL_github/refs/heads/main/Project/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv

YOUR TASK:
Provide a clear a overview of how you prepared and modelled your data. You should describe the important choices you made, especially those that shape how the model behaves or how the results should be interpreted. You do not need to repeat every step of your code. Focus instead on what a decision maker needs to know when judging whether the solution is trustworthy and practical. You can consider covering the following with relevant screenshots of code and/or visualisation, including but not limited to:
•	Transforming predictors to uncover useful patterns
•	Balancing imbalance classes, if applicable
•	Modelling choices made

Use the Job-To-Be-Done (JTBD) framework to shape your modelling objectives. State the job the user is trying to complete,
and define your target variable and likely predictors based on this job.

Generate code using Python.

"""

if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=prepare_prompt
    )
    print('\nLLM response:')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')


LLM response:
Below is an overview you can use in a decision-maker–friendly “model preparation & modelling” section. I’ll frame it with the JTBD job, explain the modelling choices that materially affect behaviour, and include Python code snippets showing key transformations and how the model is trained for each candidate year 2030–2036.

---

## 1) JTBD framing (what Mr. Chin is trying to do)

**Job-to-be-Done:**  
Mr. Chin wants to choose the *best year between 2030 and 2036* to downsize so that he **maximizes his expected net sale proceeds**, by capturing the **Cross Island Line MRT premium** while not suffering (lease-decay–driven) erosion of his flat’s value.

**Decision output needed:**  
For each candidate year \(t \in \{2030, 2031, \dots, 2036\}\), estimate **Projected Net Sale Proceeds**. Then select the maximizing year.

---

## 2) Target variable and how it maps to ML

### Primary target (regression)
For each record in the training data (historical resale transactions), the 

In [7]:
# Libraries used in this notebook
import os
import shutil
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib

# Modelling libraries used later in Session 2
from sklearn.model_selection import train_test_split, ShuffleSplit, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor

# Make wide tables easier to read in Colab
pd.set_option('display.max_columns', 100)

In [8]:
github_raw_url = "https://raw.githubusercontent.com/danielchin-ck/ADALL_github/refs/heads/main/Project/ResaleflatpricesbasedonregistrationdatefromJan2017onwards.csv"

df = pd.read_csv(github_raw_url)

print('Dataset loaded successfully.')
print('Shape:', df.shape)
display(df.head())

Dataset loaded successfully.
Shape: (237194, 11)


,month,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price
0,2017-01,ANG MO KIO,2 ROOM,406,ANG MO KIO AVE 10,10 TO 12,44.0,Improved,1979,61 years 04 months,232000.0
1,2017-01,ANG MO KIO,3 ROOM,108,ANG MO KIO AVE 4,01 TO 03,67.0,New Generation,1978,60 years 07 months,250000.0
2,2017-01,ANG MO KIO,3 ROOM,602,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,262000.0
3,2017-01,ANG MO KIO,3 ROOM,465,ANG MO KIO AVE 10,04 TO 06,68.0,New Generation,1980,62 years 01 month,265000.0
4,2017-01,ANG MO KIO,3 ROOM,601,ANG MO KIO AVE 5,01 TO 03,67.0,New Generation,1980,62 years 05 months,265000.0


In [9]:
buss_prompt = f"""

You are an expert data scientist with experience in tree-based regression models.
Help me translate this business problem into a modelling objective.

Business problem: Mr. Chin faces a criticial retirement challenge: his 36 years old Executive Apartment HDB flat is depreciating through lease decay,
but a new MRT station near his flat will be completed after 2029,potentially boosting its value. He needs to identify the optimal year within the next 5 to 10 years to downsize - balancing these opposite forces.

Dataset context: {df}

Please answer:

1. What should the modelling objective be?
2. What is the most meaningful target column?
3. Which metric would be easiest to explain to business users?
4. Who are the main stakeholders?
5. What are three risks or pitfalls?

Respond in point forms

CONSTRAINTS:
Keep it concise and to the point.
Limit each section to maximum 200 characters per line.
"""

if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=buss_prompt
    )
    print('\nLLM response:')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')


LLM response:
1) **Modelling objective**
- Predict future resale price for each flat at candidate years (next 5–10 yrs).
- Choose **optimal downsizing year** that maximizes expected sale proceeds (or net gain vs holding).

2) **Most meaningful target column**
- **resale_price** (price at observation month).
- For year-choice: model **resale_price at future year** using remaining_lease, macro time, and station-boost features.

3) **Easiest metric to explain to business users**
- **MAPE of resale_price** (% error).
- Alternative: **RMSE** (dollar error) if users prefer $ impact.

4) **Main stakeholders**
- Mr. Chin (personal decision-maker).
- HDB/real-estate advisors & mortgage/financial planners.
- Data/analytics team building the model.
- Policy/urban planners indirectly (MRT impact assumptions).

5) **Three risks / pitfalls**
- **Data leakage**: using post-event price info when predicting pre-event.
- **Non-causality**: MRT proximity correlations ≠ true station effect.
- **Time shif

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 237194 entries, 0 to 237193
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   month                237194 non-null  object 
 1   town                 237194 non-null  object 
 2   flat_type            237194 non-null  object 
 3   block                237194 non-null  object 
 4   street_name          237194 non-null  object 
 5   storey_range         237194 non-null  object 
 6   floor_area_sqm       237194 non-null  float64
 7   flat_model           237194 non-null  object 
 8   lease_commence_date  237194 non-null  int64  
 9   remaining_lease      237194 non-null  object 
 10  resale_price         237194 non-null  float64
dtypes: float64(2), int64(1), object(8)
memory usage: 19.9+ MB


In [11]:
df.columns.tolist()

['month',
 'town',
 'flat_type',
 'block',
 'street_name',
 'storey_range',
 'floor_area_sqm',
 'flat_model',
 'lease_commence_date',
 'remaining_lease',
 'resale_price']

In [12]:
data_preview = df.head(10).to_string()
print(data_preview[:1500])

     month        town flat_type block        street_name storey_range  floor_area_sqm      flat_model  lease_commence_date     remaining_lease  resale_price
0  2017-01  ANG MO KIO    2 ROOM   406  ANG MO KIO AVE 10     10 TO 12            44.0        Improved                 1979  61 years 04 months      232000.0
1  2017-01  ANG MO KIO    3 ROOM   108   ANG MO KIO AVE 4     01 TO 03            67.0  New Generation                 1978  60 years 07 months      250000.0
2  2017-01  ANG MO KIO    3 ROOM   602   ANG MO KIO AVE 5     01 TO 03            67.0  New Generation                 1980  62 years 05 months      262000.0
3  2017-01  ANG MO KIO    3 ROOM   465  ANG MO KIO AVE 10     04 TO 06            68.0  New Generation                 1980   62 years 01 month      265000.0
4  2017-01  ANG MO KIO    3 ROOM   601   ANG MO KIO AVE 5     01 TO 03            67.0  New Generation                 1980  62 years 05 months      265000.0
5  2017-01  ANG MO KIO    3 ROOM   150   ANG MO KIO 

In [13]:
preview_prompt = f"""

Here are the first 10 rows of HDB resale pricing dataset:

{data_preview}

Questions:
1. What does each row appear to represent?
2. Which column is likely the target for a price prediction model?
3. What are 3 possible data quality checks we should perform before modelling?

Keep the answer short and practical.
"""

print('Prompt to send:')
print(preview_prompt[:2000])

if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=preview_prompt
    )
    print('\nLLM response:')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

Prompt to send:


Here are the first 10 rows of HDB resale pricing dataset:

     month        town flat_type block        street_name storey_range  floor_area_sqm      flat_model  lease_commence_date     remaining_lease  resale_price
0  2017-01  ANG MO KIO    2 ROOM   406  ANG MO KIO AVE 10     10 TO 12            44.0        Improved                 1979  61 years 04 months      232000.0
1  2017-01  ANG MO KIO    3 ROOM   108   ANG MO KIO AVE 4     01 TO 03            67.0  New Generation                 1978  60 years 07 months      250000.0
2  2017-01  ANG MO KIO    3 ROOM   602   ANG MO KIO AVE 5     01 TO 03            67.0  New Generation                 1980  62 years 05 months      262000.0
3  2017-01  ANG MO KIO    3 ROOM   465  ANG MO KIO AVE 10     04 TO 06            68.0  New Generation                 1980   62 years 01 month      265000.0
4  2017-01  ANG MO KIO    3 ROOM   601   ANG MO KIO AVE 5     01 TO 03            67.0  New Generation                 1980  62 years

In [14]:
# Build payload text step by step.
# Payload text is a short profile of the dataset.
# It is safer and smaller than sending the full dataset to the LLM.

payload_text = ''

# 1. Shape
payload_text += '=== SHAPE ===\n'
payload_text += 'Rows: ' + str(df.shape[0]) + '\n'
payload_text += 'Columns: ' + str(df.shape[1]) + '\n\n'

# 2. Column names and data types
payload_text += '=== COLUMNS AND DATA TYPES ===\n'
payload_text += df.dtypes.to_string()
payload_text += '\n\n'

# 3. Numeric summary
payload_text += '=== NUMERIC SUMMARY ===\n'
numeric_summary = df.describe(include='number').round(2)
payload_text += numeric_summary.to_string()
payload_text += '\n\n'

# 4. Missing values
payload_text += '=== MISSING VALUES ===\n'
missing_table = pd.DataFrame()
missing_table['missing_count'] = df.isna().sum()
missing_table['missing_pct'] = (df.isna().sum() / len(df) * 100).round(2)
payload_text += missing_table.to_string()
payload_text += '\n\n'

# 5. Unique values per column
payload_text += '=== UNIQUE VALUES PER COLUMN ===\n'
unique_table = pd.DataFrame()
unique_table['unique_count'] = df.nunique(dropna=False)
payload_text += unique_table.to_string()
payload_text += '\n\n'

# 6. Correlation between numeric columns
payload_text += '=== CORRELATION BETWEEN NUMERIC COLUMNS ===\n'
correlation_table = df.corr(numeric_only=True).round(2)
payload_text += correlation_table.to_string()
payload_text += '\n\n'

# 7. Top 10 values for categorical columns
payload_text += '=== TOP 10 VALUES FOR CATEGORICAL COLUMNS ===\n'

categorical_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()

if len(categorical_columns) == 0:
    payload_text += 'No categorical columns found.\n'
else:
    for col in categorical_columns:
        payload_text += '\nColumn: ' + col + '\n'
        payload_text += df[col].value_counts(dropna=False).head(10).to_string()
        payload_text += '\n'

payload_text += '\n'

# 8. Simple warning checks
payload_text += '=== SIMPLE WARNING CHECKS ===\n'

id_like_columns = []
constant_columns = []

for col in df.columns:
    unique_count = df[col].nunique(dropna=False)

    if unique_count == len(df):
        id_like_columns.append(col)

    if unique_count <= 1:
        constant_columns.append(col)

payload_text += 'Possible ID-like columns: ' + str(id_like_columns) + '\n'
payload_text += 'Constant columns: ' + str(constant_columns) + '\n'
payload_text += 'Duplicate rows: ' + str(df.duplicated().sum()) + '\n'

print(payload_text)

# This cell builds a short dataset profile called payload_text.
# Instead of sending the full dataset to the LLM, we send summary information only.
# The payload includes shape, column types, numeric summary, missing values, unique counts, correlations, and common category values.
# It also adds simple warning checks for possible ID-like columns, constant columns, and duplicate rows.
# This helps the LLM comment on data readiness without needing every row of the dataset.

=== SHAPE ===
Rows: 237194
Columns: 11

=== COLUMNS AND DATA TYPES ===
month                   object
town                    object
flat_type               object
block                   object
street_name             object
storey_range            object
floor_area_sqm         float64
flat_model              object
lease_commence_date      int64
remaining_lease         object
resale_price           float64

=== NUMERIC SUMMARY ===
       floor_area_sqm  lease_commence_date  resale_price
count       237194.00            237194.00     237194.00
mean            96.69              1996.59     533134.98
std             24.02                14.38     191348.81
min             31.00              1966.00     140000.00
25%             81.00              1985.00     390000.00
50%             93.00              1997.00     500000.00
75%            112.00              2012.00     640000.00
max            366.70              2022.00    1728000.00

=== MISSING VALUES ===
                     missi

In [15]:
quality_prompt = f"""
You are helping a data analytics student prepare a HDB resale pricing dataset for modelling.

Dataset profile:
{payload_text}

Task:
1. List all data quality or modelling-readiness issues.
2. Suggest action to remedy the issues.

Important:
- Do not assume external knowledge.
- Do not say to drop a column just because it is listed as a warning.
- Keep the answer concise.
"""


if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=quality_prompt
    )
    print('\nLLM response:')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')

# This cell asks the LLM to review the dataset profile for modelling-readiness issues.
# The LLM does not receive the full dataset, only the summary stored in payload_text.
# The prompt asks for possible issues and practical remedies.
# The instructions help prevent the LLM from making unsupported assumptions or dropping columns too quickly.
# If the API client is connected, the prompt is sent automatically.
# Otherwise, students can copy the prompt and use the chatbot manually.


LLM response:
## 1) Data quality / modelling-readiness issues

1. **Duplicate rows present**
   - `Duplicate rows: 316` (no further detail given)
   - Risk: inflates sample size and can bias model training/evaluation.

2. **High-cardinality categorical features**
   - `block` (2775 unique), `street_name` (578), `month` (116), `remaining_lease` (702)
   - Risk: severe dimensionality increase (especially with one-hot encoding), sparsity, overfitting.

3. **Potentially non-numeric “lease” representation**
   - `remaining_lease` is **object** with values like `"94 years 10 months"`
   - Risk: not directly usable for numeric modelling; ordering/interval relationships may be unclear.

4. **Imperfect / ambiguous temporal fields**
   - `month` is **object** (116 unique) and there is also `lease_commence_date` (int)
   - Risk: if `month` ordering is not parsed correctly as a date/time index, time-based validation and trend features may be wrong.

5. **“Lease commence date” granularity may be c

In [16]:
quality_prompt = f"""
You are helping a data analytics student prepare a HDB resale pricing dataset for modelling.

Dataset profile:
{payload_text}

Task:
1. Provide Python code to list all duplicate rows for observation.

Important:
- Do not assume external knowledge.
- Do not say to drop a column just because it is listed as a warning.
- Keep the answer concise.
"""


if client is not None:
    response = client.responses.create(
        model=OPENAI_MODEL,
        input=quality_prompt
    )
    print('\nLLM response:')
    print(response.output_text)
else:
    print('\nNo API response because client is not connected. Copy the prompt above into your chatbot.')


LLM response:
```python
import pandas as pd

# df = your dataframe (must contain the 11 columns shown)

# Find duplicate rows (entire-row duplicates)
dup_mask = df.duplicated(keep=False)  # marks all occurrences of duplicates

dup_rows = df.loc[dup_mask].copy()

print(f"Number of duplicate-row records (including all copies): {len(dup_rows)}")
print("Duplicate rows (all columns):")
print(dup_rows)
```


In [17]:
import pandas as pd

# df = your dataframe (must contain the 11 columns shown)

# Find duplicate rows (entire-row duplicates)
dup_mask = df.duplicated(keep=False)  # marks all occurrences of duplicates

dup_rows = df.loc[dup_mask].copy()

print(f"Number of duplicate-row records (including all copies): {len(dup_rows)}")
print("Duplicate rows (all columns):")
print(dup_rows)

Number of duplicate-row records (including all copies): 631
Duplicate rows (all columns):
          month          town flat_type block       street_name storey_range  \
224     2017-01   BUKIT MERAH    4 ROOM   106    HENDERSON CRES     07 TO 09   
243     2017-01   BUKIT MERAH    4 ROOM   106    HENDERSON CRES     07 TO 09   
304     2017-01  CENTRAL AREA    3 ROOM   271          QUEEN ST     16 TO 18   
305     2017-01  CENTRAL AREA    3 ROOM   271          QUEEN ST     16 TO 18   
505     2017-01   JURONG EAST    4 ROOM   265       TOH GUAN RD     04 TO 06   
...         ...           ...       ...   ...               ...          ...   
232828  2026-03      SENGKANG    5 ROOM  224C  COMPASSVALE WALK     07 TO 09   
234808  2026-03     TOA PAYOH    4 ROOM  103B    BIDADARI PK DR     07 TO 09   
234811  2026-03     TOA PAYOH    4 ROOM  103B    BIDADARI PK DR     07 TO 09   
235207  2026-04     WOODLANDS    3 ROOM   148   WOODLANDS ST 13     01 TO 03   
235208  2026-04     WOODLANDS 

In [18]:
import pandas as pd
import numpy as np

# Make a copy to preserve the original df
cleaned_df = df.copy()

# 1. Handle duplicate rows
# Remove exact duplicate rows
initial_rows = len(cleaned_df)
cleaned_df.drop_duplicates(inplace=True)
print(f"Removed {initial_rows - len(cleaned_df)} duplicate rows.")

# 2. Reduce/encode high-cardinality categoricals appropriately
# Filter the town to 'Pasir Ris' and flat_type to 'Executive'
cleaned_df = cleaned_df[cleaned_df['town'].str.upper() == 'PASIR RIS']
cleaned_df = cleaned_df[cleaned_df['flat_type'].str.upper() == 'EXECUTIVE']
print(f"Filtered to 'Pasir Ris' town and 'Executive' flat_type. Remaining rows: {len(cleaned_df)}")

# 4. Ensure time parsing for 'month' and create 'transacted-year'
cleaned_df['month'] = pd.to_datetime(cleaned_df['month'])
cleaned_df['transaction_year'] = cleaned_df['month'].dt.year
cleaned_df['transaction_month_num'] = cleaned_df['month'].dt.month # Keep month number for potential seasonality
# As per instruction 'consider to drop [month]' and 'replace the month' with 'transacted-year'
cleaned_df.drop('month', axis=1, inplace=True)

# 3. Convert `remaining_lease` to numeric (whole years only, ignoring months)
def parse_remaining_lease_years(lease_str):
    if pd.isna(lease_str):
        return np.nan
    lease_str = str(lease_str).lower()
    years = 0
    if 'year' in lease_str:
        year_part = lease_str.split('year')[0].strip()
        try:
            years = int(year_part)
        except ValueError:
            pass # Handle cases where year_part might not be an integer
    return years

cleaned_df['remaining_lease_years'] = cleaned_df['remaining_lease'].apply(parse_remaining_lease_years)

# 5. Create “lease age (yrs)” features: years_since_commence (as whole number)
# Calculate current age of the flat at the time of transaction
# 'lease_commence_date' is int representing year, 'transaction_year' is also int
cleaned_df['age_of_flat_at_transaction'] = cleaned_df['transaction_year'] - cleaned_df['lease_commence_date']

# 8. Sanity checks for categorical consistency (trim whitespace, standardize casing)
categorical_cols_to_clean = ['town', 'flat_type', 'block', 'street_name', 'storey_range', 'flat_model']
for col in categorical_cols_to_clean:
    if col in cleaned_df.columns and cleaned_df[col].dtype == 'object':
        cleaned_df[col] = cleaned_df[col].astype(str).str.strip().str.lower()

# 7. Feature engineering for price modelling: price_per_sqm
# Check for zero floor_area_sqm to avoid division by zero
if (cleaned_df['floor_area_sqm'] == 0).any():
    print("Warning: 'floor_area_sqm' contains zero values. Handling for 'price_per_sqm' calculation.")
    cleaned_df['price_per_sqm'] = cleaned_df.apply(lambda row: row['resale_price'] / row['floor_area_sqm'] if row['floor_area_sqm'] != 0 else np.nan, axis=1)
else:
    cleaned_df['price_per_sqm'] = cleaned_df['resale_price'] / cleaned_df['floor_area_sqm']

# 6. Outlier treatment (quantify and document, but do not modify at this stage as per instruction)
print("\n--- Outlier Detection Report (No modification performed) ---")
# Only consider columns that are expected to be numeric and relevant for outlier analysis
numeric_cols_for_outliers = ['floor_area_sqm', 'resale_price', 'remaining_lease_years', 'age_of_flat_at_transaction', 'price_per_sqm']

for col in numeric_cols_for_outliers:
    if col in cleaned_df.columns:
        Q1 = cleaned_df[col].quantile(0.25)
        Q3 = cleaned_df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # Count values outside the IQR bounds, ignoring NaNs
        outliers_count = cleaned_df[(cleaned_df[col] < lower_bound) | (cleaned_df[col] > upper_bound)][col].count()
        print(f"Column '{col}': {outliers_count} outliers detected (outside 1.5*IQR). Min: {cleaned_df[col].min():.2f}, Max: {cleaned_df[col].max():.2f}")
        if outliers_count > 0:
            # Print a few examples of outliers, handle potential empty slices
            example_outliers = cleaned_df[(cleaned_df[col] < lower_bound) | (cleaned_df[col] > upper_bound)][[col]].head()
            if not example_outliers.empty:
                print(f"  Example outliers for '{col}':\n{example_outliers.to_string()}")


print("\n--- Cleaned DataFrame Info ---")
cleaned_df.info()
print("\nCleaned DataFrame head:")
print(cleaned_df.head().to_string())

Removed 316 duplicate rows.
Filtered to 'Pasir Ris' town and 'Executive' flat_type. Remaining rows: 1748

--- Outlier Detection Report (No modification performed) ---
Column 'floor_area_sqm': 35 outliers detected (outside 1.5*IQR). Min: 141.00, Max: 190.00
  Example outliers for 'floor_area_sqm':
       floor_area_sqm
705             161.0
707             165.0
7093            158.0
8947            158.0
12584           156.0
Column 'resale_price': 3 outliers detected (outside 1.5*IQR). Min: 470500.00, Max: 1260000.00
  Example outliers for 'resale_price':
        resale_price
130520     1238000.0
209856     1250000.0
229886     1260000.0
Column 'remaining_lease_years': 0 outliers detected (outside 1.5*IQR). Min: 61.00, Max: 78.00
Column 'age_of_flat_at_transaction': 0 outliers detected (outside 1.5*IQR). Min: 21.00, Max: 38.00
Column 'price_per_sqm': 2 outliers detected (outside 1.5*IQR). Min: 3179.05, Max: 8456.38
  Example outliers for 'price_per_sqm':
        price_per_sqm
209856  

In [19]:
# What to check after cleaning:
# 1. Did the number of rows stay the same?
# 2. Did the discount columns disappear?
# 3. Are there missing values that still need attention?
# 4. Does Price_SGD still exist as the target column?

print('Original rows:', df.shape[0])
print('Cleaned rows:', cleaned_df.shape[0])

print('\nMissing values after cleaning:')
missing_after_cleaning = cleaned_df.isna().sum()
display(missing_after_cleaning.sort_values(ascending=False).head(10))

print('\nColumns after cleaning:')
print(cleaned_df.columns.tolist())

Original rows: 237194
Cleaned rows: 1748

Missing values after cleaning:


,0
town,0
flat_type,0
block,0
street_name,0
storey_range,0
floor_area_sqm,0
flat_model,0
lease_commence_date,0
remaining_lease,0
resale_price,0



Columns after cleaning:
['town', 'flat_type', 'block', 'street_name', 'storey_range', 'floor_area_sqm', 'flat_model', 'lease_commence_date', 'remaining_lease', 'resale_price', 'transaction_year', 'transaction_month_num', 'remaining_lease_years', 'age_of_flat_at_transaction', 'price_per_sqm']


In [20]:
#Save to csv
cleaned_df.to_csv('cleaned_hdb_prices.csv', index=False)

print('Saved cleaned dataset as cleaned_hdb_prices.csv')


Saved cleaned dataset as cleaned_hdb_prices.csv


In [21]:
print('Loaded cleaned_hdb_prices.csv.')
print('Cleaned shape:', cleaned_df.shape)
display(cleaned_df.head())

Loaded cleaned_hdb_prices.csv.
Cleaned shape: (1748, 15)


,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,transaction_year,transaction_month_num,remaining_lease_years,age_of_flat_at_transaction,price_per_sqm
703,pasir ris,executive,239,pasir ris st 21,10 to 12,147.0,apartment,1993,75 years 07 months,552888.0,2017,1,75,24,3761.142857
704,pasir ris,executive,116,pasir ris st 11,01 to 03,155.0,maisonette,1989,71 years,598000.0,2017,1,71,28,3858.064516
705,pasir ris,executive,237,pasir ris st 21,10 to 12,161.0,apartment,1993,75 years 06 months,600000.0,2017,1,75,24,3726.708075
706,pasir ris,executive,108,pasir ris st 12,10 to 12,146.0,maisonette,1988,70 years 08 months,638000.0,2017,1,70,29,4369.863014
707,pasir ris,executive,531,pasir ris dr 1,04 to 06,165.0,maisonette,1992,74 years 11 months,728000.0,2017,1,74,25,4412.121212


In [22]:
# Function to parse storey_range and group it numerically
def get_storey_group(storey_range_str):
    if pd.isna(storey_range_str):
        return np.nan

    # The format is 'XX TO YY', convert to lower case first due to previous cleaning step
    try:
        # Extract the lower bound of the storey range
        lower_bound = int(storey_range_str.split(' to ')[0])
        # Grouping: 01-03 is group 1, 04-06 is group 2, etc.
        # (lower_bound - 1) // 3 + 1 will correctly map these
        return (lower_bound - 1) // 3 + 1
    except (ValueError, IndexError):
        return np.nan # Handle cases where parsing might fail

# Apply the function to create the new column
cleaned_df['storey_range_group'] = cleaned_df['storey_range'].apply(get_storey_group)

print("Added 'storey_range_group' column.")
print("\n--- Updated Cleaned DataFrame Info ---")
cleaned_df.info()
print("\nUpdated Cleaned DataFrame head (with 'storey_range_group'):")
print(cleaned_df[['storey_range', 'storey_range_group']].head().to_string())

Added 'storey_range_group' column.

--- Updated Cleaned DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 1748 entries, 703 to 229984
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   town                        1748 non-null   object 
 1   flat_type                   1748 non-null   object 
 2   block                       1748 non-null   object 
 3   street_name                 1748 non-null   object 
 4   storey_range                1748 non-null   object 
 5   floor_area_sqm              1748 non-null   float64
 6   flat_model                  1748 non-null   object 
 7   lease_commence_date         1748 non-null   int64  
 8   remaining_lease             1748 non-null   object 
 9   resale_price                1748 non-null   float64
 10  transaction_year            1748 non-null   int32  
 11  transaction_month_num       1748 non-null   int32  
 12  remaining_lease_

In [23]:
print('Loaded cleaned_hdb_prices.csv.')
print('Cleaned shape:', cleaned_df.shape)
display(cleaned_df.head())

Loaded cleaned_hdb_prices.csv.
Cleaned shape: (1748, 16)


,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,resale_price,transaction_year,transaction_month_num,remaining_lease_years,age_of_flat_at_transaction,price_per_sqm,storey_range_group
703,pasir ris,executive,239,pasir ris st 21,10 to 12,147.0,apartment,1993,75 years 07 months,552888.0,2017,1,75,24,3761.142857,4
704,pasir ris,executive,116,pasir ris st 11,01 to 03,155.0,maisonette,1989,71 years,598000.0,2017,1,71,28,3858.064516,1
705,pasir ris,executive,237,pasir ris st 21,10 to 12,161.0,apartment,1993,75 years 06 months,600000.0,2017,1,75,24,3726.708075,4
706,pasir ris,executive,108,pasir ris st 12,10 to 12,146.0,maisonette,1988,70 years 08 months,638000.0,2017,1,70,29,4369.863014,4
707,pasir ris,executive,531,pasir ris dr 1,04 to 06,165.0,maisonette,1992,74 years 11 months,728000.0,2017,1,74,25,4412.121212,2


In [24]:
#Start Modeling
target_col = 'resale_price'

if target_col not in cleaned_df.columns:
    raise ValueError(f'Target column {target_col} was not found. Check your cleaned dataset columns.')

X = cleaned_df.drop(columns=[target_col])
y = cleaned_df[target_col]

print('X shape:', X.shape)
print('y shape:', y.shape)
print('Target column:', target_col)
display(X.head())
display(y.head())

# This cell separates the dataset into INPUT features X and OUTPUT target y.
# target_col stores the column we want the model to predict, which is Price_SGD.
# The if-statement checks that Price_SGD still exists after data cleaning.
# X contains all columns except the target column.
# y contains only the target column.
# The shape outputs help us check how many rows and columns are being used for modelling.

X shape: (1748, 15)
y shape: (1748,)
Target column: resale_price


,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,transaction_year,transaction_month_num,remaining_lease_years,age_of_flat_at_transaction,price_per_sqm,storey_range_group
703,pasir ris,executive,239,pasir ris st 21,10 to 12,147.0,apartment,1993,75 years 07 months,2017,1,75,24,3761.142857,4
704,pasir ris,executive,116,pasir ris st 11,01 to 03,155.0,maisonette,1989,71 years,2017,1,71,28,3858.064516,1
705,pasir ris,executive,237,pasir ris st 21,10 to 12,161.0,apartment,1993,75 years 06 months,2017,1,75,24,3726.708075,4
706,pasir ris,executive,108,pasir ris st 12,10 to 12,146.0,maisonette,1988,70 years 08 months,2017,1,70,29,4369.863014,4
707,pasir ris,executive,531,pasir ris dr 1,04 to 06,165.0,maisonette,1992,74 years 11 months,2017,1,74,25,4412.121212,2


,resale_price
703,552888.0
704,598000.0
705,600000.0
706,638000.0
707,728000.0


In [25]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2, #0.2 = 20%
    random_state=42
)

print('Training rows:', X_train.shape[0])
print('Test rows:', X_test.shape[0])


Training rows: 1398
Test rows: 350


In [26]:
cat_columns = X_train.select_dtypes(include=['object', 'category']).columns.tolist()
num_columns = X_train.select_dtypes(exclude=['object', 'category']).columns.tolist()

print('Categorical columns:', cat_columns)
print('Numeric columns:', num_columns)

# This is needed because categorical and numeric columns usually need different preprocessing steps.

Categorical columns: ['town', 'flat_type', 'block', 'street_name', 'storey_range', 'flat_model', 'remaining_lease']
Numeric columns: ['floor_area_sqm', 'lease_commence_date', 'transaction_year', 'transaction_month_num', 'remaining_lease_years', 'age_of_flat_at_transaction', 'price_per_sqm', 'storey_range_group']


In [27]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_columns),
        ('num', StandardScaler(), num_columns)
    ],
    remainder='drop'
)

preprocessor

ColumnTransformer(transformers=[('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['town', 'flat_type', 'block', 'street_name',
                                  'storey_range', 'flat_model',
                                  'remaining_lease']),
                                ('num', StandardScaler(),
                                 ['floor_area_sqm', 'lease_commence_date',
                                  'transaction_year', 'transaction_month_num',
                                  'remaining_lease_years',
                                  'age_of_flat_at_transaction', 'price_per_sqm',
                                  'storey_range_group'])])

In [28]:
baseline_model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', DecisionTreeRegressor(random_state=42, max_depth=5))
])

baseline_model.fit(X_train, y_train)
print('Baseline model trained.')


Baseline model trained.


In [29]:
y_pred = baseline_model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

baseline_results = pd.DataFrame({
    'Model': ['Baseline Decision Tree'],
    'MAE': [mae],
    'RMSE': [rmse],
    'R2': [r2]
})

display(baseline_results.round(3))

# This cell uses the trained baseline model to predict laptop prices on the test set.
# y_pred stores the predicted prices.
# MAE shows the average absolute prediction error in dollars.
# RMSE also measures prediction error, but gives more penalty to large errors.
# R2 shows how much of the price variation is explained by the model.
# baseline_results stores the metrics in a simple table for comparison later.

,Model,MAE,RMSE,R2
0,Baseline Decision Tree,13868.756,20974.423,0.975


In [30]:
error_df = X_test.copy()
error_df['Actual_Price_SGD'] = y_test.values
error_df['Predicted_Price_SGD'] = y_pred.round(2)
error_df['Absolute_Error'] = np.abs(error_df['Actual_Price_SGD'] - error_df['Predicted_Price_SGD']).round(2)

# Show largest errors first
error_df_sorted = error_df.sort_values('Absolute_Error', ascending=False)
display(error_df_sorted.head(10))

# This cell creates an error analysis table for the test set.
# It copies X_test so we can view each laptop together with its prediction result.
# Actual_Price_SGD stores the true price from y_test.
# Predicted_Price_SGD stores the model's predicted price.
# Absolute_Error shows how far the prediction is from the actual price.
# Sorting by Absolute_Error helps us inspect the worst prediction mistakes first.

,town,flat_type,block,street_name,storey_range,floor_area_sqm,flat_model,lease_commence_date,remaining_lease,transaction_year,transaction_month_num,remaining_lease_years,age_of_flat_at_transaction,price_per_sqm,storey_range_group,Actual_Price_SGD,Predicted_Price_SGD,Absolute_Error
65278,pasir ris,executive,160,pasir ris st 13,07 to 09,146.0,apartment,1995,74 years 01 month,2020,1,74,25,4006.849315,3,585000.0,750000.00,165000.00
76686,pasir ris,executive,646,pasir ris dr 10,07 to 09,146.0,apartment,1995,74 years 01 month,2020,8,74,25,4041.095890,3,590000.0,750000.00,160000.00
209848,pasir ris,executive,609,elias rd,13 to 15,155.0,apartment,1995,69 years 04 months,2025,4,69,30,6806.451613,5,1055000.0,979364.84,75635.16
183978,pasir ris,executive,547,pasir ris st 51,10 to 12,154.0,maisonette,1992,67 years 02 months,2024,11,67,32,6441.558442,4,992000.0,924957.79,67042.21
156392,pasir ris,executive,237,pasir ris st 21,13 to 15,159.0,apartment,1993,69 years 04 months,2023,3,69,30,5276.025157,5,838888.0,776411.90,62476.10
229872,pasir ris,executive,601,elias rd,10 to 12,154.0,apartment,1995,68 years 03 months,2026,1,68,31,6363.636364,4,980000.0,924957.79,55042.21
130709,pasir ris,executive,241,pasir ris st 21,10 to 12,158.0,apartment,1993,69 years 08 months,2022,12,69,29,5625.873418,4,888888.0,841779.50,47108.50
93513,pasir ris,executive,420,pasir ris dr 6,01 to 03,156.0,maisonette,1989,66 years 11 months,2021,3,66,32,4788.461538,1,747000.0,700077.90,46922.10
229904,pasir ris,executive,422,pasir ris dr 6,10 to 12,151.0,maisonette,1989,62 years 05 months,2026,6,62,37,6788.079470,4,1025000.0,979364.84,45635.16
21079,pasir ris,executive,605,elias rd,07 to 09,154.0,apartment,1995,76 years 02 months,2018,1,76,23,4188.311688,3,645000.0,601225.44,43774.56
